# MENACE: a machine learning from matchboxes (1961) 🔥
Before computers were common, Donald Michie built a tic-tac-toe learner from **304 matchboxes full of coloured beads**: one box per board position, one bead colour per move. To play, shake the box and pick a bead. Win → add beads for every move made; lose → remove them. It's reinforcement learning with no maths beyond counting. Rebuild it and watch it go from random to (nearly) unbeatable against a random opponent.

Background: [[RL Basics and MDPs]], [[Q-Learning and Policy Gradients]] · overview: [[Fun Projects]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
LINES = [(0, 1, 2), (3, 4, 5), (6, 7, 8), (0, 3, 6), (1, 4, 7), (2, 5, 8), (0, 4, 8), (2, 4, 6)]

## 1. Who won?
The board is a 9-character string of `'X'`, `'O'`, `' '`. Return `'X'` or `'O'` for a winner, `'draw'` if the board is full, else `None`.

In [ ]:
def winner(board):
    # TODO 1: check the 8 lines, then full board
    raise NotImplementedError  # TODO 1

## 2. Reinforce the beads
`history` lists `(state, move)` pairs MENACE played this game. Add **3** beads of that move to each box after a win, **1** after a draw, and remove **1** after a loss (never fewer than 1 bead, so no move disappears completely).

In [ ]:
def reinforce(boxes, history, result):
    # TODO 2: update bead counts for every (state, move) in history
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_m():
    bx = {"s": {0: 2, 4: 1}}; reinforce(bx, [("s", 0)], "win"); w = bx["s"][0]; reinforce(bx, [("s", 4)], "loss")
    return winner("XXX      ") == "X" and winner("O  O  O  ") == "O" and winner("XOXXOOOXX") == "draw" and winner("X        ") is None and w == 5 and bx["s"][4] == 1
check("menace rules", _t_m, "Rows/columns/diagonals win; a full board without a line is a draw; a win adds 3 beads, a loss removes 1 but keeps ≥ 1.")

def play_game(boxes, rng, learn=True):
    board, hist = " " * 9, []
    while True:
        free = [i for i, c in enumerate(board) if c == " "]
        box = boxes.setdefault(board, {i: 4 for i in free})                       # MENACE (X) shakes the box for this position
        moves, beads = zip(*box.items()); m = rng.choice(moves, p=np.array(beads) / sum(beads))
        hist.append((board, m)); board = board[:m] + "X" + board[m + 1:]
        if winner(board): break
        o = rng.choice([i for i, c in enumerate(board) if c == " "]); board = board[:o] + "O" + board[o + 1:]   # random opponent
        if winner(board): break
    w = winner(board); result = "win" if w == "X" else "loss" if w == "O" else "draw"
    if learn: reinforce(boxes, hist, result)
    return result

if ready("menace rules"):
    rng = np.random.default_rng(0); boxes = {}; window = []
    rates = []
    for g in range(1, 4001):
        window.append(play_game(boxes, rng))
        if g % 200 == 0: rates.append((g, window.count("loss") / len(window))); window = []
    plt.plot(*zip(*rates), "o-"); plt.xlabel("games played"); plt.ylabel("loss rate (last 200)"); plt.title("MENACE vs a random player"); plt.show()
    print(f"loss rate: first 200 games {rates[0][1]:.0%} → last 200 {rates[-1][1]:.0%}; matchboxes used: {len(boxes)}")
    check("MENACE learns", lambda: rates[-1][1] < 0.7 * rates[0][1], "After 4000 games MENACE should lose at least 30 % less often than at the start.")
    print("→ 2,205 boxes is a lot to fill from 4,000 games: the symmetry stretch goal cuts that by up to 8× and speeds learning up.")

<details><summary>▶ Solution</summary>

```python
def winner(board):
    for a, b_, c in LINES:
        if board[a] != " " and board[a] == board[b_] == board[c]:
            return board[a]
    return "draw" if " " not in board else None
```

```python
def reinforce(boxes, history, result):
    delta = {"win": 3, "draw": 1, "loss": -1}[result]
    for state, move in history:
        boxes[state][move] = max(1, boxes[state][move] + delta)
```
</details>

## Stretch goals
- Use symmetry: rotations/reflections of a board share one box (Michie needed only 304). How much faster does it learn?
- Let MENACE play a perfect minimax opponent (see [[Fun Projects]] → Connect Four): can it learn to always draw?
- Compare with tabular Q-learning on the same game.